## 1.使用Message类型发送消息
在langchain中，消息统一封装为BaseMessage类。根据对应的role角色创建不同BaseMessage的子类：HumanMessage，SystemMessage，AIMessage，ToolMessage。

In [ ]:
from langchain.agents import create_agent
from langchain.tools import tool
from langchain.messages import HumanMessage,AIMessage,SystemMessage

#创建工具
@tool
def getWeather(loacation:str)->str:
    """Get the weather of a location"""
    return f"The weather of {loacation} is sunny."

#创建Agent
clint=create_agent(
    "deepseek-chat",
    tools=[getWeather]
)

#调用Agent
response=clint.invoke({
    "messages":[
        SystemMessage(content="你是妲己"),
        HumanMessage(content="江门天气怎么样?"),
        AIMessage(content="我来帮你查一下")
    ]
})

#打印一下输出
for message in response["messages"]:
    message.pretty_print()  #使用更好看的打印方法

## 2.多模态消息
- 支持多模态的模型可以去langchain官网查看，比如有openai。所以我们可以使用阿里云提供的模型，伪装成openai

### 2.1.方式一分析在线图片
传入图片url地址

In [ ]:
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from langchain.messages import HumanMessage
from dotenv import load_dotenv
import os

#加载环境
load_dotenv()
api_key=os.getenv("DASHSCOPE_API_KEY")
base_url=os.getenv("BASE_URL")

#初始化模型
model=init_chat_model(
    model="qwen3.6-plus",
    model_provider="openai",
    api_key=api_key,
    base_url=base_url
)

#创建模型
agent=create_agent(
    model=model,
)

#提供多模态信息
message = HumanMessage([
    {"type": "text", "text": "Describe the content of this image."},
    {"type": "image", "url": "https://help-static-aliyun-doc.aliyuncs.com/file-manage-files/zh-CN/20241022/emyrja/dog_and_girl.jpeg"},
]
)

#调用模型
response=agent.stream({
    "messages":[message]
},
stream_mode="messages"
)

#打印结果
for token,metadata in response:
    if token.content:
        print(token.content,end="",flush=True)




### 2.2.方式二分析本地图片
上传本地图片数据，将图片转化为base64编码，然后发送给模型。

- 写一个前端上传图片组件

In [1]:
#首先安装一个上传组件，用于模拟图片上传
#uv add ipywidgets
#然后创建一个前端上传组件
from ipywidgets import FileUpload
from IPython.display import display
uploader = FileUpload(accept='*', multiple=False)
display(uploader)



FileUpload(value=(), accept='*', description='Upload')

In [6]:
print(uploader.value)

({'name': '屏幕截图 2026-02-12 201703.png', 'type': 'image/png', 'size': 1885916, 'content': <memory at 0x0000023C196399C0>, 'last_modified': datetime.datetime(2026, 2, 12, 12, 17, 4, 19000, tzinfo=datetime.timezone.utc)},)


- 读取图片，转为base64编码

In [11]:
import base64
#获取第一个(也是唯一一个)上传的文件
uploaded_file = uploader.value[0]
# 获取其内存视图
content_mv = uploaded_file["content"]
#转换内存视图->字节
img_bytes = bytes(content_mv) # or content_mv.tobytes()
# base64编码
img_b64 = base64.b64encode(img_bytes).decode("utf-8")

In [12]:
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from langchain.messages import HumanMessage
from dotenv import load_dotenv
import os

#加载环境
load_dotenv()
api_key=os.getenv("DASHSCOPE_API_KEY")
base_url=os.getenv("BASE_URL")

#初始化模型
model=init_chat_model(
    model="qwen3.5-plus",
    model_provider="openai",
    api_key=api_key,
    base_url=base_url
)

#创建模型
agent=create_agent(
    model=model,
)




In [15]:
#提供多模态信息
message = HumanMessage([
        {
            "type": "image",
            "base64": img_b64,
            "mime_type": "image/png",
        },
        {"type": "text", "text": "描述这个图片."},
    ]
)

#调用模型
response=agent.stream({
    "messages":[message]
},
stream_mode="messages"
)

#打印结果
for token,metadata in response:
    if token.content:
        print(token.content,end="",flush=True)

这是一张电脑屏幕截图，显示的是网易武侠游戏**《燕云十六声》**的游戏启动器或登录界面。

以下是画面的详细描述：

**1. 背景画面**
*   **场景**：背景是一个昏暗、充满雾气的中国古代宫殿内部场景。
*   **细节**：可以看到深色的石板地面，右侧有一张长案（书案），上面摆放着卷轴和笔筒。地面铺着带有龙纹图案的紫色圆形地毯。周围立着几个烛台，发出微弱的光芒。远处有红色的柱子和栏杆，营造出一种肃穆、神秘的氛围。
*   **左上角**：有巨大的书法字体，虽然被截断，但能辨认出是游戏标题“燕云十六声”的一部分。

**2. 左侧新闻/公告面板**
*   **轮播图**：最上方是一张雪景古建筑的宣传图，标题写着**“【开封皇宫】数据同步现已上线”**，副标题提到“一键查询宝箱/蹊跷，助力极速探索100%”。图片上方有“燕云十六声 x 大神 x 网易DD”的联名标识。
*   **列表**：下方列出了几条公告，分为“活动”、“公告”、“资讯”三个标签页。
    *   第一条：2月8日 2.4.5版本「宫阙日初升」... 日期显示为 **2026-02-08**（这个年份显示非常奇怪，可能是显示错误或私服/测试服特征）。
    *   第二条：2月11日 2.4.5版本在线修复公告 ... 日期显示为 **2026-02-11**。
    *   第三条：2月6日违规玩家封禁公告 ... 日期显示为 **2026-02-06**。
    *   第四条：常见下载、安装问题指引 ... 日期显示为 **2024-12-27**。
*   **侧边**：最左侧有竖排文字“江湖要闻”。

**3. 底部下载状态栏**
*   **左下角**：显示下载信息。
    *   剩余时间：**124:42:08**（超过5天，时间非常长）。
    *   进度：**资源下载中 0.00%**。
    *   数据量：已下载 8.81M / 总大小 **188.51G**（这个包体非常大，远超普通游戏更新）。
    *   速度：0.43M/s。
*   **右下角**：有一个灰色的按钮显示**“下载暂停”**，旁边有版本号 `1.4.0.42967`。

**4. 顶部功能栏**
*   右上角有一排社交和功能图标，包括微信、大神（网易社区）、抖音、Bi